# Create a `BOW` and `DTCM` table for the Brandon Sanderson Dataset

## Get Data to build `BOW` table

In [1]:
import pandas as pd
import numpy as np

directory_path  = 'C:/Users/mason/Box/Text As Data Final/Output' 

In [2]:
CORPUS = pd.read_csv(directory_path + "/BrandonSanderson_CORPUS.csv", index_col=0)
CORPUS.head()

,title,chapter_id,paragraph_id,sent_id,token_id,token_str,term_str,pos,pos_group
0,A Memory of Light,0,0,0,0,The,the,DT,DT
1,A Memory of Light,0,0,0,1,Wheel,wheel,NNP,NN
2,A Memory of Light,0,0,0,2,of,of,IN,IN
3,A Memory of Light,0,0,0,3,Time,time,NNP,NN
4,A Memory of Light,0,0,0,4,turns,turns,NNS,NN


## Set the OHCO

In [3]:
OHCO = ['title', 'chapter_id', 'paragraph_id', 'sent_id', 'token_id']
bags = dict(
    SENTS = OHCO[:4],
    PARAS = OHCO[:3],
    CHAPS = OHCO[:2],
    BOOKS = OHCO[:1]
)
bag = 'CHAPS'

## Create the TOKEN table from CORPUS and save the OHCO

In [4]:
TOKEN = CORPUS.set_index(OHCO).dropna()
bags[bag]+['term_str']

['title', 'chapter_id', 'term_str']

## Get the term counts,`n`, per bag


In [5]:
BOW = TOKEN.groupby(bags[bag]+['term_str']).term_str.count().to_frame('n')
BOW.head()

n
title             chapter_id term_str      
A Memory of Light 0          a          141
                             abandoned    1
                             abilities    1
                             ability      1
                             able         4

# Create the DTCM a sparse version of the BOW

In [6]:
DTCM = BOW.n.unstack(fill_value=0)
DTCM.head(10)

term_str                      0  04  05  1  10  10010  10584  109thand  11  \
title             chapter_id                                                 
A Memory of Light 0           0   0   0  0   0      0      0         0   0   
                  1           0   0   0  0   0      0      0         0   0   
                  2           0   0   0  0   0      0      0         0   0   
                  3           0   0   0  0   0      0      0         0   0   
                  4           0   0   0  0   0      0      0         0   0   
                  5           0   0   0  0   0      0      0         0   0   
                  6           0   0   0  0   0      0      0         0   0   
                  7           0   0   0  0   0      0      0         0   0   
                  8           0   0   0  0   0      0      0         0   0   
                  9           0   0   0  0   0      0      0         0   0   

term_str                      1167  ...  ﬂowers  ﬂuidity  ﬂush  ﬂushed  \
title             chapter_id        ...                                  
A Memory of Light 0              0  ...       0        0     0       0   
                  1              0  ...       0        0     0       0   
                  2              0  ...       0        0     0       0   
                  3              0  ...       0        0     0       0   
                  4              0  ...       0        0     0       0   
                  5              0  ...       0        0     0       0   
                  6              0  ...       0        0     0       0   
                  7              0  ...       0        0     0       0   
                  8              0  ...       0        0     0       0   
                  9              0  ...       0        0     0       0   

term_str                      ﬂushedi  ﬂushing  ﬂushingi  ﬂushingit  \
title             chapter_id                                          
A Memory of Light 0                 0        0         0          0   
                  1                 0        0         0          0   
                  2                 0        0         0          0   
                  3                 0        0         0          0   
                  4                 0        0         0          0   
                  5                 0        0         0          0   
                  6                 0        0         0          0   
                  7                 0        0         0          0   
                  8                 0        0         0          0   
                  9                 0        0         0          0   

term_str                      ﬂuttering  ﬂying  
title             chapter_id                    
A Memory of Light 0                   0      0  
                  1                   0      0  
                  2                   0      0  
                  3                   0      0  
                  4                   0      0  
                  5                   0      0  
                  6                   0      0  
                  7                   0      0  
                  8                   0      0  
                  9                   0      0  

[10 rows x 58423 columns]

In [7]:
DTCM.to_csv(directory_path + "/BrandonSanderson_DTCM.csv")

## Get the `TFIDF` per bag

In [8]:
total_words_per_bag = BOW.groupby(bags[bag])['n'].sum()
BOW['tf'] = BOW['n'] / total_words_per_bag

In [9]:
DF = BOW.groupby('term_str')['n'].count()
N = len(BOW.index.droplevel('term_str').unique())
IDF = np.log2(N / DF)

In [10]:
BOW['idf'] = IDF
BOW['tfidf'] = BOW['tf'] * BOW['idf']
BOW.head()

n        tf       idf     tfidf
title             chapter_id term_str                                    
A Memory of Light 0          a          141  0.017229  0.013379  0.000231
                             abandoned    1  0.000122  3.174371  0.000388
                             abilities    1  0.000122  3.321928  0.000406
                             ability      1  0.000122  2.246264  0.000274
                             able         4  0.000489  0.586073  0.000286

## Save the output as `BOW`

In [11]:
BOW.to_csv(directory_path + "/BrandonSanderson_BOW.csv")

In [12]:
len(BOW)

638071

In [13]:
columns = BOW.columns.tolist()
columns

['n', 'tf', 'idf', 'tfidf']